# Gold | Fato-Chamado


In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("gold_table_name", "fato_chamado")
gold_table_name = dbutils.widgets.get("gold_table_name")

dbutils.widgets.text("silver_chamados_table_name", "tb_chamados")
silver_chamados_table_name = dbutils.widgets.get("silver_chamados_table_name")

dbutils.widgets.text("silver_custos_table_name", "tb_custos")
silver_custos_table_name = dbutils.widgets.get("silver_custos_table_name")

dbutils.widgets.text("silver_notas_table_name", "tb_pesquisas_de_satisfacao")
silver_notas_table_name = dbutils.widgets.get("silver_notas_table_name")

dbutils.widgets.text("gold_cliente_table_name", "dim_cliente")
gold_cliente_table_name = dbutils.widgets.get("gold_cliente_table_name")

dbutils.widgets.text("gold_atendente_table_name", "dim_atendente")
gold_atendente_table_name = dbutils.widgets.get("gold_atendente_table_name")

dbutils.widgets.text("gold_canal_table_name", "dim_canal")
gold_canal_table_name = dbutils.widgets.get("gold_canal_table_name")

dbutils.widgets.text("gold_motivo_table_name", "dim_motivo")
gold_motivo_table_name = dbutils.widgets.get("gold_motivo_table_name")

dbutils.widgets.text("gold_data_table_name", "dim_data")
gold_data_table_name = dbutils.widgets.get("gold_data_table_name")

dbutils.widgets.text("gold_horario_table_name", "dim_horario")
gold_horario_table_name = dbutils.widgets.get("gold_horario_table_name")


In [0]:
df_chamado = (
    load_df(silver_chamados_table_name)
        .select(
            "id_chamado", "id_cliente", "canal", "motivo",
            # id_atendente é string na dim_atendente; o cast explícito evita depender de cast implícito
            F.col("id_atendente").cast("string").alias("id_atendente"),
            "resolvido",
            "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento",
            "tempo_espera_segundos", "tempo_atendimento_segundos"
        )
)
print(f"Total de linhas de chamado: {df_chamado.count()}")

In [0]:
# Custo por chamado (no máximo um custo por chamado na origem; a soma evita duplicar linhas)
df_custos = (
    load_df(silver_custos_table_name)
        .groupBy("id_chamado")
        .agg(F.sum("custo").cast("decimal(18,2)").alias("custo_chamado"))
)

# Nota por chamado (as pesquisas repetidas do mesmo chamado trazem a mesma nota)
df_notas = (
    load_df(silver_notas_table_name)
        .groupBy("id_chamado")
        .agg(F.max("nota_atendimento").alias("nota_atendimento"))
)

In [0]:
gold_schema = f"{dbutils.widgets.get('catalog')}.gold"

df_dim_cliente = load_df(gold_cliente_table_name, gold_schema).select("cliente_sk", "id_cliente")
df_dim_atendente = (
    load_df(gold_atendente_table_name, gold_schema)
        .select("atendente_sk", F.col("id_atendente").cast("string").alias("id_atendente"))
)
df_dim_canal = load_df(gold_canal_table_name, gold_schema).select("canal_sk", F.col("nome_canal").alias("canal"))
df_dim_motivo = load_df(gold_motivo_table_name, gold_schema).select("motivo_sk", F.col("nome_motivo").alias("motivo"))
df_dim_horario = (
    load_df(gold_horario_table_name, gold_schema)
        .select(F.col("horario_sk").cast("int").alias("horario_sk"))
)

# A chave da dimensão de data já é yyyyMMdd; só o cast de bigint para int é necessário
df_dim_data = (
    load_df(gold_data_table_name, gold_schema)
        .select(F.col("data_sk").cast("int").alias("data_sk"))
)

# Tratamento



In [0]:
# Junta chamado com horários, custos e notas (por id_chamado)
df_medidas = (
    df_chamado
    .join(df_custos, "id_chamado", "left")
    .join(df_notas, "id_chamado", "left")
)
print(f"Total de linhas de horários: {df_medidas.count()}")

In [0]:
df_dims = (
    df_medidas
    .join(df_dim_cliente, "id_cliente", "left")
    .join(df_dim_atendente, "id_atendente", "left")
    .join(df_dim_canal, "canal", "left")
    .join(df_dim_motivo, "motivo", "left")
    .withColumn("data_sk", F.date_format("hora_abertura_chamado", "yyyyMMdd").cast("int"))
)

df_dims = df_dims.join(df_dim_data, "data_sk", "left")


In [0]:
# Busca a chave de horário (formato HHmmss) de uma coluna de timestamp
def adicionar_horario_sk(df, coluna, nome_sk):
    return (
        df.withColumn(nome_sk, F.date_format(coluna, "HHmmss").cast("int"))
          .join(df_dim_horario.select(F.col("horario_sk").alias(nome_sk)), nome_sk, "left")
    )


In [0]:
df_horario_sk = adicionar_horario_sk(df_dims, "hora_abertura_chamado", "horario_abertura_sk")
df_horario_sk = adicionar_horario_sk(df_horario_sk, "hora_inicio_atendimento", "horario_inicio_sk")
df_horario_sk = adicionar_horario_sk(df_horario_sk, "hora_finalizacao_atendimento", "horario_final_sk")

In [0]:
janela_seg = 7 * 86400

df_base_cadeia = (
    df_horario_sk
    .filter(F.col("cliente_sk").isNotNull())
    .select(
        "id_chamado",
        F.col("cliente_sk").alias("c_cliente"),
        F.coalesce(F.col("motivo_sk"), F.lit(-1)).alias("c_motivo"),
        F.col("hora_abertura_chamado").cast("timestamp").cast("long").alias("c_abertura"),
        F.col("hora_finalizacao_atendimento").cast("timestamp").cast("long").alias("c_fim"),
        F.col("resolvido").cast("int").alias("c_resolvido")
    )
)

w_seq = Window.partitionBy("c_cliente", "c_motivo").orderBy("c_abertura", "id_chamado")

df_base_cadeia = (
    df_base_cadeia
    .withColumn("c_prox", F.lead("c_abertura").over(w_seq))
    .withColumn("c_ant", F.lag("c_abertura").over(w_seq))
    .withColumn(
        "gerou_rechamada",
        F.coalesce(((F.col("c_prox") - F.col("c_abertura")) <= janela_seg).cast("int"), F.lit(0))
    )
    .withColumn(
        "c_inicio",
        F.when(F.col("c_ant").isNull() | ((F.col("c_abertura") - F.col("c_ant")) > janela_seg), 1).otherwise(0)
    )
    .withColumn("c_num", F.sum("c_inicio").over(w_seq.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
)

w_cad = Window.partitionBy("c_cliente", "c_motivo", "c_num").orderBy("c_abertura", "id_chamado")
w_cad_total = w_cad.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df_cadeia = (
    df_base_cadeia
    .withColumn("ordem_na_cadeia", F.row_number().over(w_cad))
    .withColumn("cadeia_id", F.first("id_chamado").over(w_cad_total))
    .withColumn("c_primeira_abertura", F.first("c_abertura").over(w_cad_total))
    .withColumn("c_ultimo_fim", F.max("c_fim").over(w_cad_total))
    .withColumn("c_resolvido_final", F.last("c_resolvido", ignorenulls=True).over(w_cad_total))
    .withColumn(
        "tempo_resolucao_seg",
        F.when(
            (F.col("ordem_na_cadeia") == 1) & (F.col("c_resolvido_final") == 1),
            (F.col("c_ultimo_fim") - F.col("c_primeira_abertura")).cast("int")
        )
    )
    .select("id_chamado", "cadeia_id", "ordem_na_cadeia", "gerou_rechamada", "tempo_resolucao_seg")
)

df_horario_sk = df_horario_sk.join(df_cadeia, "id_chamado", "left")

In [0]:
# id_chamado mantém o tipo da silver (int), sem cast que possa anular valores
df_fato = df_horario_sk.select(
    "id_chamado",
    F.coalesce(F.col("cliente_sk").cast("int"), F.lit(-1)).alias("cliente_sk"),
    F.coalesce(F.col("motivo_sk").cast("int"), F.lit(-1)).alias("motivo_sk"),
    F.coalesce(F.col("canal_sk").cast("int"), F.lit(-1)).alias("canal_sk"),
    F.coalesce(F.col("data_sk").cast("int"), F.lit(-1)).alias("data_sk"),
    F.coalesce(F.col("horario_abertura_sk").cast("int"), F.lit(-1)).alias("horario_abertura_sk"),
    F.coalesce(F.col("horario_inicio_sk").cast("int"), F.lit(-1)).alias("horario_inicio_sk"),
    F.coalesce(F.col("horario_final_sk").cast("int"), F.lit(-1)).alias("horario_final_sk"),
    F.coalesce(F.col("atendente_sk").cast("int"), F.lit(-1)).alias("atendente_sk"),
    F.col("resolvido").cast("int").alias("resolvido"),
    F.col("tempo_espera_segundos").cast("int").alias("tempo_espera"),
    F.col("tempo_atendimento_segundos").cast("int").alias("tempo_atendimento"),
    "custo_chamado",  # Mantido decimal para precisão monetária
    F.col("nota_atendimento").cast("int").alias("nota_atendimento"),
    "cadeia_id",
    F.col("ordem_na_cadeia").cast("int").alias("ordem_na_cadeia"),
    F.col("gerou_rechamada").cast("int").alias("gerou_rechamada"),
    F.col("tempo_resolucao_seg").cast("int").alias("tempo_resolucao_seg")
).orderBy("id_chamado")

## Conferências

In [0]:
df_tempos_origem = load_df(silver_chamados_table_name).select("id_chamado", "tempo_espera_segundos")

divergentes = (
    df_fato.join(df_tempos_origem.withColumnRenamed("tempo_espera_segundos", "tempo_espera_origem"), "id_chamado")
        .filter(F.col("tempo_espera_origem").isNotNull() & (F.abs(F.col("tempo_espera_origem") - F.col("tempo_espera")) > 1))
        .count()
)

print(f"chamado com tempo de espera divergente entre as duas fontes: {divergentes}")

print(f"chamado sem cliente na dimensão (SK = -1): {df_fato.filter(F.col('cliente_sk') == -1).count()}")

print(f"chamado sem custo: {df_fato.filter(F.col('custo_chamado').isNull()).count()}")
print(f"chamado sem nota: {df_fato.filter(F.col('nota_atendimento').isNull()).count()}")

In [0]:
dq_check_unique("fato_chamado", "id_chamado único", df_fato, ["id_chamado"])

In [0]:
# Nenhum chamado da silver pode ter ficado de fora da fato
df_perdidos = df_chamado.join(df_fato.select("id_chamado"), "id_chamado", "left_anti")

dq_check(
    "fato_chamado",
    "todo chamado da silver está na fato",
    df_perdidos,
    F.lit(False)
)

# Como as chaves agora assumem -1 quando não há conexão, essa validação garante que nenhuma chave ficou NULL
dq_check(
    "fato_chamado",
    "chaves obrigatórias preenchidas",
    df_fato,
    F.col("motivo_sk").isNotNull()
    & F.col("canal_sk").isNotNull()
    & F.col("data_sk").isNotNull()
    & F.col("horario_abertura_sk").isNotNull()
    & F.col("horario_final_sk").isNotNull()
    & F.col("cliente_sk").isNotNull()
    & F.col("atendente_sk").isNotNull()
    & F.col("horario_inicio_sk").isNotNull()
)

In [0]:
dq_check(
    "fato_chamado",
    "tempos não podem ser negativos",
    df_fato,
    (F.col("tempo_espera").isNull() | (F.col("tempo_espera") >= 0))
    & (F.col("tempo_atendimento").isNull() | (F.col("tempo_atendimento") >= 0))
)

In [0]:
dq_check(
    "fato_chamado",
    "nota_atendimento entre 1 e 5",
    df_fato,
    F.col("nota_atendimento").isNull() | F.col("nota_atendimento").between(1, 5)
)

# Salvamento

In [0]:
save_df(df_fato, gold_table_name)

## Validação final

In [0]:
df_gold = load_df(gold_table_name, gold_schema)

print(f"Linhas na gold: {df_gold.count()}")
display(df_gold.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_gold.columns]))
display(df_gold.groupBy("canal_sk").count().orderBy("canal_sk"))

In [0]:
# Contagem de registros sem conexão (SK=-1) por dimensão
sk_cols = [
    "cliente_sk", "motivo_sk", "canal_sk", "data_sk", 
    "horario_abertura_sk", "horario_inicio_sk", "horario_final_sk", "atendente_sk"
]

print("Quantidade de registros sem conexão (SK=-1) por relação")
display(df_fato.select([
    F.sum((F.col(col) == -1).cast("int")).alias(col) for col in sk_cols
]))